In [1]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files={'train': 'cleaned_mixed_data_no_punctuation2.csv'}, split='train')
dataset = dataset.train_test_split(test_size=0.2)


In [2]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert/distilbert-base-uncased")

# Preprocess function
def preprocess(batch):
    return tokenizer(batch['Text'], padding="max_length", truncation=True, max_length=128)

# Apply to dataset
encoded_dataset = dataset.map(preprocess, batched=True)


Map:   0%|          | 0/6400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

In [3]:
encoded_dataset = encoded_dataset.rename_column("Label", "labels")

In [4]:
encoded_dataset

DatasetDict({
    train: Dataset({
        features: ['Text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6400
    })
    test: Dataset({
        features: ['Text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1600
    })
})

In [5]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
encoded_dataset.keys()

dict_keys(['train', 'test'])

In [7]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

training_args = TrainingArguments(
    output_dir="./distilbert-speaksense",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


C:\Users\Rohit Francis\AppData\Local\Temp\ipykernel_24208\1909889686.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.004804,0.998750,0.999177,0.999177,0.999177
2,0.039000,0.007013,0.998750,0.999176,1.000000,0.998354
3,0.000900,0.001290,0.999375,0.999588,1.000000,0.999177


TrainOutput(global_step=1200, training_loss=0.016649123035992187, metrics={'train_runtime': 258.5638, 'train_samples_per_second': 74.256, 'train_steps_per_second': 4.641, 'total_flos': 635843513548800.0, 'train_loss': 0.016649123035992187, 'epoch': 3.0})

In [4]:
import torch
print(torch.cuda.is_available())  # Should be True
print(torch.cuda.get_device_name(0))  # Print GPU name


True
NVIDIA GeForce RTX 3050 Laptop GPU


In [1]:
from transformers import pipeline

classifier = pipeline("text-classification", model="./distilbert-speaksense/checkpoint-800", tokenizer="./distilbert-speaksense/checkpoint-1200")
classifier("I gotta have my lunch and after that go to work again, what do you think I should do after taht")
# Output: [{'label': 'LABEL_1', 'score': 0.9876}]


Device set to use cuda:0


[{'label': 'LABEL_0', 'score': 0.9993131160736084}]

In [2]:
print(classifier("I gotta have my lunch and after that go to work again, what do you think I should do after taht"))
print(classifier("I was planning to make a really good martini"))
print(classifier("I was planning to make a really good martini, do you like martini?"))
print(classifier("So I'm going to take an event on...on quantum computing. I only know little about..."))
print(classifier("about it by the little I mean...really little like I only made some...A small game."))

[{'label': 'LABEL_0', 'score': 0.9993131160736084}]
[{'label': 'LABEL_1', 'score': 0.9937663078308105}]
[{'label': 'LABEL_0', 'score': 0.9993365406990051}]
[{'label': 'LABEL_1', 'score': 0.6117749810218811}]
[{'label': 'LABEL_1', 'score': 0.9980087876319885}]


In [5]:
print(classifier("for levantics. Previousец fun. denk-down. Alone ...How to get you a fixer ... And design ...So ... But to say ... My true thinking what to say ...Do you have any opinion on any food that ..... I should try ... It seems like you're trying to convey ..."))


[{'label': 'LABEL_0', 'score': 0.9996638298034668}]
